In [1]:
import pandas as pd
import json 


In [3]:
# Tarik file kecil
df_game = pd.read_csv('../steam.csv/steam.csv')

# Kita cuma butuh ID, Nama, dan Harga buat meng analisa bisnis
df_katalog = df_game[['appid', 'name', 'price', 'developer']].copy()

# Hapus data kosong
df_katalog = df_katalog.dropna()

print(f"Total game di Master Data: {len(df_katalog)} judul")
# Tunjukan 3 baris pertama untuk verifikasi 
df_katalog.head(3)

Total game di Master Data: 27074 judul


,appid,name,price,developer
0,10,Counter-Strike,7.19,Valve
1,20,Team Fortress Classic,3.99,Valve
2,30,Day of Defeat,3.99,Valve


In [4]:
# Tarik data steam reviews (3GB), test 10.000 baris dulu untuk testing!
# Path-nya ngikutin pola folder lu yang tadi (folder -> file)
df_review = pd.read_csv('../steam_reviews.csv/steam_reviews.csv', nrows=10000)

print(f"Berhasil tarik {len(df_review)} baris review dari file steam_reviews.")
# Tunjukin 3 baris pertama biar yakin datanya bener
df_review.head(5)

Berhasil tarik 10000 baris review dari file steam_reviews.


,Unnamed: 0,app_id,app_name,review_id,language,review,timestamp_created,timestamp_updated,recommended,votes_helpful,...,steam_purchase,received_for_free,written_during_early_access,author.steamid,author.num_games_owned,author.num_reviews,author.playtime_forever,author.playtime_last_two_weeks,author.playtime_at_review,author.last_played
0,0,292030,The Witcher 3: Wild Hunt,85185598,schinese,不玩此生遗憾，RPG游戏里的天花板，太吸引人了,1611381629,1611381629,True,0,...,True,False,False,76561199095369542,6,2,1909.0,1448.0,1909.0,1.611343e+09
1,1,292030,The Witcher 3: Wild Hunt,85185250,schinese,拔DIAO无情打桩机--杰洛特!!!,1611381030,1611381030,True,0,...,True,False,False,76561198949504115,30,10,2764.0,2743.0,2674.0,1.611386e+09
2,2,292030,The Witcher 3: Wild Hunt,85185111,schinese,巫师3NB,1611380800,1611380800,True,0,...,True,False,False,76561199090098988,5,1,1061.0,1061.0,1060.0,1.611384e+09
3,3,292030,The Witcher 3: Wild Hunt,85184605,english,"One of the best RPG's of all time, worthy of a...",1611379970,1611379970,True,0,...,True,False,False,76561199054755373,5,3,5587.0,3200.0,5524.0,1.611384e+09
4,4,292030,The Witcher 3: Wild Hunt,85184287,schinese,大作,1611379427,1611379427,True,0,...,True,False,False,76561199028326951,7,4,217.0,42.0,217.0,1.610788e+09


In [5]:
import pandas as pd

# 1. PERSIAPAN DATA UNTUK MYSQL
print("Memproses data untuk MySQL...")

# Membaca dataset utama dari folder
df_katalog = pd.read_csv('../steam.csv/steam.csv')

# Memilih kolom spesifik yang relevan untuk skema database relasional
kolom_mysql = ['appid', 'name', 'release_date', 'price', 'developer', 'publisher']
df_mysql = df_katalog[kolom_mysql].copy()

# Menghapus baris yang memiliki nilai kosong (Null/NaN) untuk mencegah error saat import database
df_mysql = df_mysql.dropna()

# Menyimpan hasil pembersihan ke file CSV baru
df_mysql.to_csv('clean_mysql_katalog.csv', index=False)
print("File 'clean_mysql_katalog.csv' berhasil dibuat.")


# 2. PERSIAPAN DATA UNTUK MONGODB
print("Memproses data untuk MongoDB...")

# Membaca dataset ulasan. Menggunakan parameter usecols untuk membatasi kolom agar tidak membebani memori RAM.
# Menggunakan nrows=500000 untuk membatasi pemrosesan pada 500 ribu baris pertama untuk keperluan progress report.
kolom_mongo = ['app_id', 'app_name', 'review_id', 'language', 'recommended', 'author.playtime_forever']
df_review = pd.read_csv('../steam_reviews.csv/steam_reviews.csv', usecols=kolom_mongo, nrows=500000)

# Menyeragamkan nama kolom primary key ('app_id' menjadi 'appid') agar sesuai dengan tabel MySQL
df_review = df_review.rename(columns={'app_id': 'appid'})

# Menyimpan data ulasan ke file CSV baru. MongoDB Compass mendukung import langsung dari format CSV.
df_review.to_csv('clean_mongodb_reviews.csv', index=False)
print("File 'clean_mongodb_reviews.csv' berhasil dibuat.")

Memproses data untuk MySQL...
File 'clean_mysql_katalog.csv' berhasil dibuat.
Memproses data untuk MongoDB...
File 'clean_mongodb_reviews.csv' berhasil dibuat.


In [6]:
import pandas as pd
from IPython.display import display

# Membaca file bersih yang baru saja diekstrak
df_mysql_cek = pd.read_csv('clean_mysql_katalog.csv')
df_mongo_cek = pd.read_csv('clean_mongodb_reviews.csv')

print("=== BUKTI DATA MYSQL (KATALOG) ===")
print(f"Total baris: {len(df_mysql_cek)}")
# Menampilkan 5 baris pertama dalam bentuk tabel interaktif Jupyter
display(df_mysql_cek.head())

print("\n=== BUKTI DATA MONGODB (ULASAN) ===")
print(f"Total baris: {len(df_mongo_cek)}")
# Menampilkan 5 baris pertama dalam bentuk tabel interaktif Jupyter
display(df_mongo_cek.head())

=== BUKTI DATA MYSQL (KATALOG) ===
Total baris: 27061


,appid,name,release_date,price,developer,publisher
0,10,Counter-Strike,2000-11-01,7.19,Valve,Valve
1,20,Team Fortress Classic,1999-04-01,3.99,Valve,Valve
2,30,Day of Defeat,2003-05-01,3.99,Valve,Valve
3,40,Deathmatch Classic,2001-06-01,3.99,Valve,Valve
4,50,Half-Life: Opposing Force,1999-11-01,3.99,Gearbox Software,Valve



=== BUKTI DATA MONGODB (ULASAN) ===
Total baris: 500000


,appid,app_name,review_id,language,recommended,author.playtime_forever
0,292030,The Witcher 3: Wild Hunt,85185598,schinese,True,1909.0
1,292030,The Witcher 3: Wild Hunt,85185250,schinese,True,2764.0
2,292030,The Witcher 3: Wild Hunt,85185111,schinese,True,1061.0
3,292030,The Witcher 3: Wild Hunt,85184605,english,True,5587.0
4,292030,The Witcher 3: Wild Hunt,85184287,schinese,True,217.0
